# Dia 14 — O export do grupo 🧬

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-14-json-xml-csv](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-14-json-xml-csv)

## 🎬 Missão

> O colaborador arrependido do Dia 13 fez mais uma: exportou o grupo **interno**
> da equipe da PixVelox antes de ser removido. Um `export-grupo.json` de 26 KB,
> com membros, 120 mensagens e uma surpresa.
>
> — O maré_alta apaga tudo o que escreve de importante — avisa o colaborador.
>
> Bento rola o arquivo e para numa linha:
> — `"apagada": true`... e logo embaixo, `"texto_original"`. O app escondeu da
> tela, mas o export levou junto. Clássico.
>
> No mesmo pacote vieram o `sitemap.xml` do site falso e um CSV de domínios
> parecidos com o da PixVelox.

**Sua tarefa:** usar o `jq` para filtrar, entre as 120 mensagens, as que
foram **apagadas pelo maré_alta**, recuperar o texto original delas na ordem
e montar a flag `VAZANTE{...}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### Os três formatos de dados que você mais vai ver

```text
JSON: o formato das APIs e dos exports de apps (WhatsApp, Telegram, Discord).
XML:  sitemaps, feeds RSS, arquivos de configuração, documentos Office.
CSV:  planilhas, bases de dados abertos, vazamentos.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "O que chegou da perícia"
ls -lh desafio/
head -c 400 desafio/export-grupo.json; echo
head -n 4 desafio/sitemap.xml
head -n 3 desafio/dominios.csv

### jq: o primeiro filtro

```text
`jq` lê JSON e aplica um FILTRO. O ponto (.) é "o documento inteiro".
  jq '.'                  imprime bonito e colorido (pretty print)
  jq '.grupo'             entra na chave "grupo"
  jq '.grupo.nome'        chave dentro de chave
  jq -r                   "raw": tira as aspas das strings
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "jq: navegando por chaves"
precisa jq
jq '.grupo' desafio/export-grupo.json
jq -r '.grupo.criado_por' desafio/export-grupo.json

### jq: listas (arrays)

```text
  .membros[0]             o primeiro item (começa do zero!)
  .membros[0].handle      a chave "handle" do primeiro membro
  .membros[].handle       a chave "handle" de TODOS os membros
  .membros | length       quantos itens a lista tem
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "jq: índices e [ ]"
precisa jq
jq '.membros[0].handle' desafio/export-grupo.json
jq -r '.membros[].handle' desafio/export-grupo.json
jq '.mensagens | length' desafio/export-grupo.json

### jq: select (filtrar) e map (transformar)

```text
`select(condição)` deixa passar só os itens em que a condição é verdadeira.
`map(f)` aplica f a cada item de uma lista e devolve outra lista.
O cano `|` dentro do jq funciona como no Bash: passa o resultado adiante.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "jq: select e map"
precisa jq
echo "--- membros que são admin:"
jq -r '.membros[] | select(.admin) | .handle' desafio/export-grupo.json
echo "--- as 3 primeiras mensagens do lucas_suporte:"
jq -r '.mensagens[] | select(.autor.handle == "lucas_suporte") | "\(.data)  \(.texto)"' desafio/export-grupo.json | head -n 3
echo "--- map: só os ids das 5 primeiras mensagens"
jq -c '.mensagens[:5] | map(.id)' desafio/export-grupo.json

### jq: contando e agrupando

```text
Quem mais fala no grupo? group_by agrupa, length conta, sort_by ordena.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "Ranking de mensagens por autor"
precisa jq
jq -r '.mensagens | group_by(.autor.handle) | map({autor: .[0].autor.handle, total: length})
       | sort_by(-.total) | .[] | "\(.total)\t\(.autor)"' desafio/export-grupo.json
echo "--- mensagens apagadas no export:"
jq '[.mensagens[] | select(.apagada)] | length' desafio/export-grupo.json
dica "Mensagem apagada que ainda tem texto_original no export... interessante, não?"

### jq: transformando JSON em CSV

```text
@csv converte um array em linha CSV. Ótimo para levar para a planilha.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "JSON -> CSV com jq"
precisa jq
jq -r '.membros[] | [.handle, .nome_exibido, .admin] | @csv' desafio/export-grupo.json

### xmlstarlet: consultando XML

```text
`xmlstarlet sel` (select) usa XPath para achar elementos:
  -t                 começa um "template" de saída
  -m "//TITLE"       para cada elemento TITLE em qualquer lugar (match)
  -v .               imprime o valor do elemento atual
  -n                 quebra de linha
Sitemaps usam um "namespace" (xmlns=...). No xmlstarlet o atalho _: resolve.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "xmlstarlet: as URLs do sitemap"
precisa xmlstarlet
xmlstarlet sel -t -m "//_:url" -v "_:loc" -o "  prioridade=" -v "_:priority" -n desafio/sitemap.xml
echo "--- só as páginas com prioridade baixa (< 0.5):"
xmlstarlet sel -t -m "//_:url[_:priority < 0.5]" -v "_:loc" -n desafio/sitemap.xml

### xmlstarlet sem namespace (como no curso original)

```text
Num XML sem xmlns, o XPath é direto: //TITLE.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "XML simples"
precisa xmlstarlet
mkdir -p treino
printf '<CATALOG><CD><TITLE>Maré Alta</TITLE></CD><CD><TITLE>Vazante</TITLE></CD></CATALOG>\n' > treino/xml_sample.xml
xmlstarlet sel -t -m "//TITLE" -v . -n treino/xml_sample.xml
rm -r treino

### csvkit: CSV com ferramentas que entendem cabeçalho e aspas

```text
O `cut` (Dia 10) quebra quando um campo tem vírgula dentro de aspas.
O csvkit entende CSV de verdade.
  csvlook          mostra como tabela
  csvcut -c a,b    escolhe colunas pelo NOME
  csvgrep -c 2 -m texto    filtra linhas em que a coluna 2 contém "texto"
  csvgrep -c porta -r '^80$'   com expressão regular
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "csvkit"
precisa_pip csvlook csvkit
csvlook desafio/dominios.csv
echo "--- só domínio e porta:"
csvcut -c dominio,porta desafio/dominios.csv
echo "--- títulos que mencionam PixVelox (coluna 2):"
csvgrep -d ',' -c 2 -m 'PixVelox' desafio/dominios.csv

### in2csv: JSON (e Excel) -> CSV

```text
  in2csv -k membros arq.json   converte a lista que está na chave "membros"
  in2csv planilha.xlsx         também converte Excel!
  csvstat                      estatísticas por coluna
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "in2csv e csvstat"
precisa_pip in2csv csvkit
mkdir -p treino
in2csv -k membros desafio/export-grupo.json > treino/membros.csv
cat treino/membros.csv
csvcut -c dominio,porta desafio/dominios.csv > treino/cut_csv.csv
csvstat -c porta treino/cut_csv.csv | head -n 12
rm -r treino

### Dados reais de uma API (online)

```text
APIs públicas devolvem JSON — e o jq é o par perfeito do curl (Dia 5).
Exemplo: dados públicos de um CEP pela BrasilAPI (01001000 = Praça da Sé, SP).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-14-json-xml-csv" && source "$L4O_HOME/lib/comum.sh"
titulo "curl + jq numa API pública"
precisa jq
if online; then
  curl -s -m 10 https://brasilapi.com.br/api/cep/v1/01001000 | jq '{cep, city, neighborhood, street}'
else
  alerta "offline: usando cópia salva"
  jq '{cep, city, neighborhood, street}' desafio/cep-01001000.json
fi

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-14-json-xml-csv" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-14-json-xml-csv" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)